In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, RandomizedSearchCV, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.linear_model import Ridge

from xgboost import XGBRegressor
def get_catboost_regressor():
    try:
        from catboost import CatBoostRegressor
        return CatBoostRegressor
    except Exception as _e:
        print('CatBoost not available:', _e)
        return None

from scipy.stats import randint, uniform, loguniform

RANDOM_STATE = 42

FAST_RUN = True
N_ITER = 5 if FAST_RUN else 25
CV_SPLITS = 3 if FAST_RUN else 5
SEARCH_JOBS = 1 if FAST_RUN else -1
CV_JOBS = 1 if FAST_RUN else -1


In [2]:
train_path = "train.csv"
test_path  = "test.csv"

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

print(train_df.shape, test_df.shape)
train_df.head()


(15000, 10) (10000, 9)


,id,Sex,Length,Diameter,Height,Weight,Shucked Weight,Viscera Weight,Shell Weight,Age
0,0,F,1.6000,1.2500,0.5750,43.700754,15.025235,10.262519,13.324265,17.0
1,1,M,1.5750,1.1625,0.3750,32.417653,12.927372,6.208540,8.930093,20.0
2,2,F,1.6125,1.2750,0.4375,34.373769,14.784264,9.142714,10.772810,10.0
3,3,M,1.2250,0.9875,0.3250,18.455524,8.037083,3.926406,4.819415,9.0
4,4,I,1.3750,1.0250,0.3875,22.736299,9.879801,5.513978,6.945627,10.0


In [3]:
TARGET = "Age"
ID_COL = "id"

assert TARGET in train_df.columns, "Donot find target in train!"
assert TARGET not in test_df.columns, "Donot find target in test!"

X = train_df.drop(columns=[TARGET])
y = train_df[TARGET].astype(float)

print("Train columns:", list(train_df.columns))
print("Missing values (train):\n", train_df.isna().sum())
print("dtypes:\n", train_df.dtypes)


Train columns: ['id', 'Sex', 'Length', 'Diameter', 'Height', 'Weight', 'Shucked Weight', 'Viscera Weight', 'Shell Weight', 'Age']
Missing values (train):
 id                0
Sex               0
Length            0
Diameter          0
Height            0
Weight            0
Shucked Weight    0
Viscera Weight    0
Shell Weight      0
Age               0
dtype: int64
dtypes:
 id                  int64
Sex                object
Length            float64
Diameter          float64
Height            float64
Weight            float64
Shucked Weight    float64
Viscera Weight    float64
Shell Weight      float64
Age               float64
dtype: object


In [6]:
drop_cols = [ID_COL] if ID_COL in X.columns else []
X = X.drop(columns=drop_cols)
X_test = test_df.drop(columns=drop_cols)

cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

print("Categorical cols:", cat_cols)
print("Numeric cols:", num_cols)


Categorical cols: ['Sex']
Numeric cols: ['Length', 'Diameter', 'Height', 'Weight', 'Shucked Weight', 'Viscera Weight', 'Shell Weight']


In [8]:
def build_onehot():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", build_onehot())
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
    remainder="drop"
)


In [ ]:
rf = RandomForestRegressor(
    random_state=RANDOM_STATE,
    n_jobs=-1
)

xgb = XGBRegressor(
    objective="reg:squarederror",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    tree_method="hist"
)

CatBoostRegressor = get_catboost_regressor()

cat = None
if CatBoostRegressor is not None:
    cat = CatBoostRegressor(
        loss_function="MAE",
        random_seed=RANDOM_STATE,
        verbose=0
    )


In [10]:
rf_params = {
    "model__n_estimators": randint(400, 1600),
    "model__max_depth": [None, 6, 10, 14, 20, 30],
    "model__min_samples_split": randint(2, 12),
    "model__min_samples_leaf": randint(1, 6),
    "model__max_features": ["sqrt", "log2", 0.5, 0.8],
}

xgb_params = {
    "model__n_estimators": randint(600, 3000),
    "model__learning_rate": loguniform(0.01, 0.2),
    "model__max_depth": randint(3, 11),
    "model__subsample": uniform(0.6, 0.4),
    "model__colsample_bytree": uniform(0.6, 0.4),
    "model__min_child_weight": randint(1, 11),
    "model__gamma": uniform(0.0, 5.0),
    "model__reg_lambda": loguniform(0.5, 10.0),
}

cat_params = {
    "model__iterations": randint(1500, 5000),
    "model__learning_rate": loguniform(0.01, 0.2),
    "model__depth": randint(4, 11),
    "model__l2_leaf_reg": loguniform(1.0, 10.0),
    "model__subsample": uniform(0.6, 0.4),
    "model__rsm": uniform(0.6, 0.4),
}


In [11]:
def tune_model(base_model, param_distributions, X, y, n_iter=25):
    pipe = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", base_model)
    ])

    cv = KFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    search = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=param_distributions,
        n_iter=n_iter,
        scoring="neg_mean_absolute_error",  
        cv=cv,
        n_jobs=SEARCH_JOBS,
        verbose=1,
        random_state=RANDOM_STATE
    )
    search.fit(X, y)
    print("Best MAE:", -search.best_score_)
    print("Best params:", search.best_params_)
    return search.best_estimator_


In [ ]:
best_rf  = tune_model(rf,  rf_params,  X, y, n_iter=N_ITER)
best_xgb = tune_model(xgb, xgb_params, X, y, n_iter=N_ITER)
best_cat = None
if cat is not None:
    best_cat = tune_model(cat, cat_params, X, y, n_iter=N_ITER)


Fitting 5 folds for each of 25 candidates, totalling 125 fits
Best MAE: 1.3643512268520213
Best params: {'model__max_depth': 10, 'model__max_features': 0.5, 'model__min_samples_leaf': 2, 'model__min_samples_split': 4, 'model__n_estimators': 1231}
Fitting 5 folds for each of 25 candidates, totalling 125 fits
Best MAE: 1.357328693675995
Best params: {'model__colsample_bytree': np.float64(0.7836995567863468), 'model__gamma': np.float64(1.668543055695109), 'model__learning_rate': np.float64(0.015341718702799927), 'model__max_depth': 5, 'model__min_child_weight': 6, 'model__n_estimators': 1369, 'model__reg_lambda': np.float64(4.3482255528158475), 'model__subsample': np.float64(0.9754210836063001)}
Fitting 5 folds for each of 25 candidates, totalling 125 fits


In [ ]:
estimators = [("rf", best_rf), ("xgb", best_xgb)]
if best_cat is not None:
    estimators.append(("cat", best_cat))

stack = StackingRegressor(
    estimators=estimators,
    final_estimator=Ridge(alpha=1.0),
    passthrough=False,
    n_jobs=CV_JOBS
)

cv = KFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    "mae": "neg_mean_absolute_error",
    "r2": "r2"
}

scores = cross_validate(stack, X, y, cv=cv, scoring=scoring, n_jobs=CV_JOBS)

mae = -scores["test_mae"]
r2  = scores["test_r2"]

print("MAE folds:", mae)
print("MAE mean:", mae.mean(), "MAE std:", mae.std())
print("R2 mean:", r2.mean(), "R2 std:", r2.std())


In [ ]:
stack.fit(X, y)
test_pred = stack.predict(X_test)

test_pred[:10], test_pred.mean()
